In [ ]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.interpolate import griddata
from scipy.linalg import lstsq
from scipy.ndimage import median_filter

from skimage import transform

import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [2]:
def lsm_read(lsm_file):
    # for reading in lsm output from ZEISS Confomap

    df = pd.read_csv(lsm_file,names=['x','y','z'])

    x = np.asarray(df['x'])
    y = np.asarray(df['y'])
    z = np.asarray(df['z'])

    # calculate shape - this must be done on the raw data 
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)


    # create new grid to interpolate data onto
    xg,yg = np.meshgrid(np.arange(x0,x1,x_step),np.arange(y0,y1,y_step))



    # remove the weird way that ConfoMaps saves non-measured points
    x = x[z !='***']
    y = y[z !='***']
    z = z[z !='***']


    # interpolate onto grid to produced gridded data
    zg = griddata(np.asarray([x,y]).T,z,(xg,yg),method='nearest')

    # flip up down for zg
    zg = np.flipud(zg)

    return xg, yg, zg, x_step

def resample_gridded_data(xg,yg,zg,new_step):

    # flatten arrays - we could probably use RegularGridInterpolator but this works for now
    x = xg.flatten()
    y = yg.flatten()
    z = zg.flatten()

    # calculate shape - this must be done on the raw data
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)

    # create new grid to interpolate data onto
    xg_new,yg_new = np.meshgrid(np.arange(x0,x1,new_step),np.arange(y0,y1,new_step))

    # interpolate onto grid to produced gridded data
    zg_new = griddata(np.asarray([x,y]).T,z,(xg_new,yg_new),method='nearest')

    return xg_new, yg_new, zg_new


In [3]:

# path to LSM file 
lsm_file = './LSM/raw_surface.txt'

# import data
xg,yg,zg,lsm_step = lsm_read(lsm_file)




# processing to get data into useful form 
# crop the rubbish data from edges 
xL = 800
xR = 800
yT = 800
yB = 800

# if we want to nanify the deleted data
# zg[:,:xL] = np.nan
# zg[:,-xR:] = np.nan

# zg[:yT,:] = np.nan
# zg[-yB:,:] = np.nan

# plt.figure()
# plt.imshow(zg)

zg = zg[yT:-yB,xL:-xR]
xg = xg[yT:-yB,xL:-xR]
yg = yg[yT:-yB,xL:-xR]
# plt.figure()
# plt.imshow(zg)



# best-fit linear plane
A = np.c_[xg.flatten(),yg.flatten(), np.ones(xg.flatten().shape[0])]

C,_,_,_ = lstsq(A, zg.flatten())    # coefficients
    
# fitted plane
zg_fit = C[0]*xg + C[1]*yg + C[2]

# plt.figure()
fig,ax = plt.subplots(1,3)
ax[0].imshow(zg)
ax[0].set_title('Raw surface')
ax[1].imshow(zg_fit)
ax[1].set_title('Fitted flat plane')
ax[2].imshow(zg - zg_fit)
ax[2].set_title('Corrected')

plt.tight_layout()

# corrected surface
zg_flat = zg - zg_fit

# set lowest point on map to zero 
zg_flat = zg_flat - zg_flat.min()

In [4]:
# current dataset is overkill for DIC
# resample to DIC step size 

dic_step_px = 10
dic_px_size = 20/2048
dic_step = dic_step_px*dic_px_size # microns 



xg_new, yg_new, zg_new = resample_gridded_data(xg,yg,zg_flat,dic_step)



In [5]:
# median filter to smooth out bumps from speckle pattern

# filter kernel size
k_size = 15

zg_filtered = median_filter(zg_new,k_size)



In [10]:
fig,ax = plt.subplots()
surf = ax.imshow(zg_filtered,cmap='gist_grey',vmin=0,vmax=2)
bar = plt.colorbar(surf)
bar.set_label('Altitude / μm')

In [28]:
zg_grad = np.gradient(zg_filtered)
zg_grad_mag = ((zg_grad[0]**2 + zg_grad[1]**2)**0.5)/dic_step

zg_grad_mag = median_filter(zg_grad_mag,20)

In [33]:
fig,ax = plt.subplots()
surf = ax.imshow(zg_grad_mag,cmap='gist_grey',vmin=0,vmax=0.1)
bar = plt.colorbar(surf)
bar.set_label('Altitude gradient magnitude / μm/μm')

In [9]:
exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

# for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
#     hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

dic_file = dic_step_list[-2]
hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)


In [91]:
# clip points on DIC map - start bottom left and go ACW

fig,ax = plt.subplots(1,2)
surf = ax[0].imshow(zg_grad_mag,cmap='gist_grey',vmin=0,vmax=0.1)
bar = plt.colorbar(surf)
bar.set_label('Altitude gradient magnitude / μm/μm')


ax[1].imshow(dic_map.data['max_shear'],vmin=0,vmax=0.1)

dic_click = plt.ginput(9,show_clicks=True,timeout=-1)
dic_click = np.asarray(dic_click)

ax[1].plot(dic_click[:,0],dic_click[:,1],'rx')

In [92]:
# clip points on zmap map - start bottom left and go ACW

fig,ax = plt.subplots(1,2)


ax[1].imshow(dic_map.data['max_shear'],vmin=0,vmax=0.1)
ax[1].plot(dic_click[:,0],dic_click[:,1],'rx')

surf = ax[0].imshow(zg_grad_mag,cmap='gist_grey',vmin=0,vmax=0.1)
bar = plt.colorbar(surf)
bar.set_label('Altitude gradient magnitude / μm/μm')

lsm_click = plt.ginput(9,show_clicks=True,timeout=-1)
lsm_click = np.asarray(lsm_click)

ax[0].plot(lsm_click[:,0],lsm_click[:,1],'rx')

In [94]:
dic_click

array([[ 213.8016129 ,  193.61013105],
       [1348.15      ,  277.8000504 ],
       [2619.8608871 ,  171.45488911],
       [ 523.975     , 2072.37464718],
       [1591.85766129, 1389.99319556],
       [2504.65362903, 1257.06174395],
       [ 178.35322581, 2692.72142137],
       [1645.03024194, 2918.70488911],
       [2885.72379032, 2728.16980847]])

In [96]:
lsm_click = [[ 284.30426747,  316.98132981],
       [1423.52133737,  357.66765373],
       [2801.04401882,  142.61137013],
       [ 534.23454301, 2083.93025454],
       [1656.01461694, 1409.69974378],
       [2580.17540323, 1316.70243196],
       [ 202.93161962, 2676.78811744],
       [1824.57224462, 2990.65404486],
       [2969.60164651, 2717.47444136]]

dic_click = [[ 213.8016129 ,  193.61013105],
       [1348.15      ,  277.8000504 ],
       [2619.8608871 ,  171.45488911],
       [ 523.975     , 2072.37464718],
       [1591.85766129, 1389.99319556],
       [2504.65362903, 1257.06174395],
       [ 178.35322581, 2692.72142137],
       [1645.03024194, 2918.70488911],
       [2885.72379032, 2728.16980847]]

In [97]:
tf = transform.ProjectiveTransform()
tf.estimate(lsm_click,dic_click)

zg_grad_mag_warped = transform.warp(zg_grad_mag,tf.inverse,output_shape=dic_map.shape)
zg_filtered_warped = transform.warp(zg_filtered,tf.inverse,output_shape=dic_map.shape)

# we need to shift this to account for the crop in the DIC map
tf_shift = transform.AffineTransform(translation=(100,100))

zg_grad_mag_warped = transform.warp(zg_grad_mag_warped,tf_shift.inverse,output_shape=dic_map.shape)
zg_filtered_warped = transform.warp(zg_filtered_warped,tf_shift.inverse,output_shape=dic_map.shape)


C:\Users\bepoole\AppData\Local\Temp\ipykernel_20692\244900557.py:2: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  tf.estimate(lsm_click,dic_click)


In [98]:
# surely it can't be this simple! 
# let's try to bodge it in to defdap 
dic_map.data.add(
    'lsm_height2', zg_filtered_warped,
    unit='μm', type='map', order=0,
    plot_params={
        'plot_colour_bar': True,
        'clabel': 'Height',
    }
)


In [71]:

# link ebsd  map
ebsd_frame = experiment.Frame()
data_dir = Path('.')
ebsd.Map(data_dir / 'Pre_EBSD/map.cpr',
         increment=exp.increments[0], frame=ebsd_frame)

ebsd_map = exp.increments[0].maps['ebsd']
# ebsd_map.set_homog_point()

dic_map = exp.increments[0].maps['hrdic']

# dic_map.set_homog_point(vmin=0,vmax=0.05)

ebsd_frame.homog_points = [(1946, 1565),
 (2443, 1000),
 (1305, 1027),
 (1395, 2225),
 (2572, 2193),
 (1876, 1077),
 (2613, 1497),
 (1822, 2208),
 (1259, 1661),
 (2229, 1260),
 (1641, 1325),
 (1693, 1794),
 (2195, 1762)]

dic_frame.homog_points = [(1582, 1380),
 (2615, 172),
 (238, 226),
 (453, 2748),
 (2882, 2728),
 (1431, 326),
 (2970, 1240),
 (1329, 2732),
 (157, 1568),
 (2170, 731),
 (941, 863),
 (1061, 1854),
 (2100, 1795)]

ebsd_map = exp.increments[0].maps['ebsd']

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.link_ebsd_map(ebsd_map, transform_type="polynomial",order=2)
    # dic_map.link_ebsd_map(ebsd_map, transform_type="affine")

Loaded EBSD data (dimensions: 3727 x 2795 pixels, step size: 0.2 um)


In [108]:
h_plot = dic_map.plot_map('lsm_height2',plot_gbs='line',plot_scale_bar=True,boundary_colour='black',cmap='gist_earth')

C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,


In [109]:
ms_plot = dic_map.plot_map('max_shear')

In [112]:

dic_map.calc_line_profile(ms_plot,(1550,1515,1660,1400))

In [ ]:
fig,ax = plt.subplot
dic_map.calc_line_profile(h_plot,(1550,1515,1660,1400),fig=fig,ax=ax)

In [ ]:
# surface map 
# fig,ax = plt.subplots(subplot_kw={"projection":"3d"})

# rcount = 1000

# surf = ax.plot_surface(xg_new, yg_new, zg_new,rcount=rcount, ccount=rcount,cmap='terrain')
# ax.set_box_aspect((np.ptp(xg_new),np.ptp(yg_new),np.ptp(zg_new)))
# ax.set_zlim(0,2)
# fig.colorbar(surf)

